# ML - Base semanal de demanda - Lojas físicas

Objetivo:
Criar uma base histórica semanal de vendas físicas para servir como entrada futura do modelo de previsão de demanda.

In [0]:
%run "../config/00_config"

In [0]:
%run "../utils/00_utils"

In [0]:
# Define imports, caminhos e parâmetros da base semanal de demanda.

from pyspark.sql.functions import (
    col,
    count,
    current_timestamp,
    date_add,
    date_sub,
    dayofweek,
    max as spark_max,
    min as spark_min,
    round as spark_round,
    sum as spark_sum,
    to_date,
    weekofyear,
    year,
    month,
    when
)

SILVER_VENDAS_TABLE = "physical_vendas_caixa"
SILVER_VENDAS_PATH = f"{SILVER_BASE_PATH}{SILVER_VENDAS_TABLE}"

SILVER_FERIADOS_TABLE = "feriados"
SILVER_FERIADOS_PATH = f"{SILVER_BASE_PATH}{SILVER_FERIADOS_TABLE}"

GOLD_DOMAIN = "physical_vendas_caixa"
GOLD_KPI = "demanda_historica_semanal"

GOLD_PATH = f"{GOLD_BASE_PATH}{GOLD_DOMAIN}/{GOLD_KPI}"

GOLD_WRITE_MODE = "overwrite"

VENDAS_REQUIRED_COLUMNS = [
    "id_transacao",
    "dt_venda",
    "valor_total_venda"
]

FERIADOS_REQUIRED_COLUMNS = [
    "data_feriado",
    "nome_feriado",
    "tipo_feriado",
    "categoria_feriado"
]

GOLD_KEY_COLUMNS = [
    "data_inicio_semana"
]

adls_options = get_adls_options()

print("Parâmetros definidos com sucesso.")
print("SILVER_VENDAS_PATH:", SILVER_VENDAS_PATH)
print("SILVER_FERIADOS_PATH:", SILVER_FERIADOS_PATH)
print("GOLD_PATH:", GOLD_PATH)

In [0]:
# Lê a Silver de vendas físicas e valida colunas obrigatórias.

df_vendas = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(SILVER_VENDAS_PATH)
)

validate_required_columns(df_vendas, VENDAS_REQUIRED_COLUMNS)

total_vendas = df_vendas.count()

print("Silver de physical_vendas_caixa lida com sucesso.")
print(f"Total de registros: {total_vendas}")

df_vendas.printSchema()

display(df_vendas.limit(10))

In [0]:
# Verifica o período histórico disponível para previsão.

df_periodo = (
    df_vendas
    .agg(
        spark_min("dt_venda").alias("menor_data_venda"),
        spark_max("dt_venda").alias("maior_data_venda"),
        count("id_transacao").alias("qtd_transacoes")
    )
)

display(df_periodo)

In [0]:
# Prepara as vendas físicas com data e início da semana.

from pyspark.sql.functions import date_trunc

df_vendas_semana_base = (
    df_vendas
    .withColumn("data_venda", to_date(col("dt_venda")))
    .withColumn("data_inicio_semana", to_date(date_trunc("week", col("dt_venda"))))
    .withColumn("data_fim_semana", date_add(col("data_inicio_semana"), 6))
)

display(
    df_vendas_semana_base
    .select(
        "id_transacao",
        "dt_venda",
        "data_venda",
        "data_inicio_semana",
        "data_fim_semana",
        "valor_total_venda"
    )
    .orderBy("dt_venda")
    .limit(20)
)

In [0]:
# Agrega as vendas físicas por semana.

df_vendas_semanais = (
    df_vendas_semana_base
    .groupBy(
        "data_inicio_semana",
        "data_fim_semana"
    )
    .agg(
        count("id_transacao").alias("qtd_transacoes"),
        spark_sum("valor_total_venda").alias("receita_total")
    )
    .withColumn(
        "ticket_medio",
        spark_round(col("receita_total") / col("qtd_transacoes"), 2)
    )
    .withColumn("ano", year(col("data_inicio_semana")))
    .withColumn("mes", month(col("data_inicio_semana")))
    .withColumn("semana_ano", weekofyear(col("data_inicio_semana")))
    .orderBy("data_inicio_semana")
)

display(df_vendas_semanais)

In [0]:
# Valida se a agregação semanal manteve o total de transações.

total_original = df_vendas.count()

total_semanal = (
    df_vendas_semanais
    .agg(spark_sum("qtd_transacoes").alias("total_transacoes"))
    .collect()[0]["total_transacoes"]
)

print("Total original:", total_original)
print("Total semanal:", total_semanal)

if total_original == total_semanal:
    print("Validação OK: os totais batem.")
else:
    print("Atenção: os totais não batem.")

In [0]:
# Lê a Silver de feriados.

df_feriados = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(SILVER_FERIADOS_PATH)
)

validate_required_columns(df_feriados, FERIADOS_REQUIRED_COLUMNS)

print("Silver de feriados lida com sucesso.")
print(f"Total de feriados: {df_feriados.count()}")

df_feriados.printSchema()

display(df_feriados.limit(10))

In [0]:
# Agrupa feriados por semana.

from pyspark.sql.functions import coalesce, lit, date_trunc

df_feriados_semanais = (
    df_feriados
    .withColumn("data_inicio_semana", to_date(date_trunc("week", col("data_feriado"))))
    .groupBy("data_inicio_semana")
    .agg(
        count("data_feriado").alias("qtd_feriados_semana")
    )
)

display(
    df_feriados_semanais
    .orderBy("data_inicio_semana")
)

In [0]:
# Junta vendas semanais com informação de feriados.

df_demanda_historica_semanal = (
    df_vendas_semanais
    .join(
        df_feriados_semanais,
        on="data_inicio_semana",
        how="left"
    )
    .withColumn(
        "qtd_feriados_semana",
        coalesce(col("qtd_feriados_semana"), lit(0))
    )
    .withColumn(
        "tem_feriado_semana",
        when(col("qtd_feriados_semana") > 0, lit(1)).otherwise(lit(0))
    )
    .withColumn("gold_processed_at", current_timestamp())
    .orderBy("data_inicio_semana")
)

display(df_demanda_historica_semanal)

In [0]:
# Valida a base histórica semanal com feriados.

total_original = df_vendas.count()

total_base_semanal = (
    df_demanda_historica_semanal
    .agg(spark_sum("qtd_transacoes").alias("total_transacoes"))
    .collect()[0]["total_transacoes"]
)

qtd_semanas = df_demanda_historica_semanal.count()

qtd_semanas_com_feriado = (
    df_demanda_historica_semanal
    .filter(col("tem_feriado_semana") == 1)
    .count()
)

print("Total original:", total_original)
print("Total base semanal:", total_base_semanal)
print("Quantidade de semanas:", qtd_semanas)
print("Semanas com feriado:", qtd_semanas_com_feriado)

if total_original == total_base_semanal:
    print("Validação OK: os totais continuam batendo.")
else:
    print("Atenção: os totais não batem.")

In [0]:
# Adiciona canal e semana do mês.

from pyspark.sql.functions import ceil, dayofmonth

df_demanda_historica_semanal_final = (
    df_demanda_historica_semanal
    .withColumn("canal", lit("lojas_fisicas"))
    .withColumn(
        "semana_mes",
        ceil(dayofmonth(col("data_inicio_semana")) / lit(7))
    )
    .select(
        "data_inicio_semana",
        "data_fim_semana",
        "ano",
        "mes",
        "semana_ano",
        "semana_mes",
        "canal",
        "qtd_transacoes",
        "receita_total",
        "ticket_medio",
        "qtd_feriados_semana",
        "tem_feriado_semana",
        "gold_processed_at"
    )
    .orderBy("data_inicio_semana")
)

display(df_demanda_historica_semanal_final)

In [0]:
# Valida se existe mais de uma linha por semana.

df_duplicadas = (
    df_demanda_historica_semanal_final
    .groupBy("data_inicio_semana")
    .agg(count("*").alias("qtd_linhas"))
    .filter(col("qtd_linhas") > 1)
)

qtd_duplicadas = df_duplicadas.count()

print("Semanas duplicadas:", qtd_duplicadas)

if qtd_duplicadas == 0:
    print("Validação OK: não há semanas duplicadas.")
else:
    print("Atenção: existem semanas duplicadas.")
    display(df_duplicadas)

In [0]:
# Valida nulos nas colunas principais da base final.

from pyspark.sql.functions import sum as spark_sum

df_validacao_nulos = (
    df_demanda_historica_semanal_final
    .select(
        spark_sum(col("data_inicio_semana").isNull().cast("int")).alias("nulos_data_inicio_semana"),
        spark_sum(col("data_fim_semana").isNull().cast("int")).alias("nulos_data_fim_semana"),
        spark_sum(col("qtd_transacoes").isNull().cast("int")).alias("nulos_qtd_transacoes"),
        spark_sum(col("receita_total").isNull().cast("int")).alias("nulos_receita_total"),
        spark_sum(col("ticket_medio").isNull().cast("int")).alias("nulos_ticket_medio"),
        spark_sum(col("canal").isNull().cast("int")).alias("nulos_canal")
    )
)

display(df_validacao_nulos)

In [0]:
# Remove a primeira e a última semana do histórico, pois podem estar incompletas.

from pyspark.sql.functions import date_trunc

df_limites_periodo = (
    df_vendas
    .agg(
        spark_min("dt_venda").alias("menor_data_venda"),
        spark_max("dt_venda").alias("maior_data_venda")
    )
    .collect()[0]
)

menor_data_venda = df_limites_periodo["menor_data_venda"]
maior_data_venda = df_limites_periodo["maior_data_venda"]

primeira_semana_historico = (
    df_vendas
    .select(to_date(date_trunc("week", col("dt_venda"))).alias("data_inicio_semana"))
    .agg(spark_min("data_inicio_semana").alias("primeira_semana"))
    .collect()[0]["primeira_semana"]
)

ultima_semana_historico = (
    df_vendas
    .select(to_date(date_trunc("week", col("dt_venda"))).alias("data_inicio_semana"))
    .agg(spark_max("data_inicio_semana").alias("ultima_semana"))
    .collect()[0]["ultima_semana"]
)

df_demanda_historica_semanal_gold = (
    df_demanda_historica_semanal_final
    .filter(col("data_inicio_semana") > lit(primeira_semana_historico))
    .filter(col("data_inicio_semana") < lit(ultima_semana_historico))
    .orderBy("data_inicio_semana")
)

print("Menor data de venda:", menor_data_venda)
print("Maior data de venda:", maior_data_venda)
print("Primeira semana removida:", primeira_semana_historico)
print("Última semana removida:", ultima_semana_historico)

print("Semanas antes:", df_demanda_historica_semanal_final.count())
print("Semanas após remoção:", df_demanda_historica_semanal_gold.count())

display(df_demanda_historica_semanal_gold)

In [0]:
# Valida a base final após remoção das semanas incompletas.

total_transacoes_base_completa = (
    df_demanda_historica_semanal_final
    .agg(spark_sum("qtd_transacoes").alias("total_transacoes"))
    .collect()[0]["total_transacoes"]
)

total_transacoes_base_treino = (
    df_demanda_historica_semanal_gold
    .agg(spark_sum("qtd_transacoes").alias("total_transacoes"))
    .collect()[0]["total_transacoes"]
)

qtd_semanas_base_completa = df_demanda_historica_semanal_final.count()
qtd_semanas_base_treino = df_demanda_historica_semanal_gold.count()

print("Total de transações antes:", total_transacoes_base_completa)
print("Total de transações após remoção:", total_transacoes_base_treino)
print("Semanas antes:", qtd_semanas_base_completa)
print("Semanas após remoção:", qtd_semanas_base_treino)
print("Transações removidas:", total_transacoes_base_completa - total_transacoes_base_treino)

In [0]:
# Grava a base histórica semanal de demanda na camada Gold.

(
    df_demanda_historica_semanal_gold
    .write
    .format("delta")
    .mode(GOLD_WRITE_MODE)
    .option("overwriteSchema", "true")
    .options(**adls_options)
    .partitionBy("ano", "mes")
    .save(GOLD_PATH)
)

print("Gold gravada com sucesso.")
print("Destino ADLS:", GOLD_PATH)

In [0]:
# Lê a Gold gravada para validação.

df_gold_validacao = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(GOLD_PATH)
)

print("Gold lida com sucesso.")
print(f"Total de linhas gravadas: {df_gold_validacao.count()}")

df_gold_validacao.printSchema()

display(
    df_gold_validacao
    .orderBy("data_inicio_semana")
)

In [0]:
# Valida se a Gold gravada preservou a base final usada para treino.

total_gold = (
    df_gold_validacao
    .agg(spark_sum("qtd_transacoes").alias("total_transacoes"))
    .collect()[0]["total_transacoes"]
)

total_base_treino = (
    df_demanda_historica_semanal_gold
    .agg(spark_sum("qtd_transacoes").alias("total_transacoes"))
    .collect()[0]["total_transacoes"]
)

qtd_linhas_gold = df_gold_validacao.count()
qtd_linhas_base_treino = df_demanda_historica_semanal_gold.count()

print("Total base treino:", total_base_treino)
print("Total Gold:", total_gold)
print("Linhas base treino:", qtd_linhas_base_treino)
print("Linhas Gold:", qtd_linhas_gold)

if total_base_treino == total_gold and qtd_linhas_base_treino == qtd_linhas_gold:
    print("Validação OK: a Gold foi gravada corretamente.")
else:
    print("Atenção: a Gold gravada não bate com a base final.")